# 캐럿·컷·색상·투명도로 다이아몬드 가격을 얼마나 설명할 수 있는가

## 1. 분석 개요

### 분석 주제

공개 데이터인 Diamonds 데이터셋을 이용해 **다이아몬드의 특성이 가격과 어떤 관련이 있으며, 가격 차이를 얼마나 설명할 수 있는지** 확인한다. 캐럿·컷·색상·투명도를 중심으로 데이터를 살펴보고, 선형회귀의 전체 과정인 품질 점검 → 탐색적 분석 → 모델링 → 진단 → 해석을 연습한다.

이 분석에서 `price`는 다이아몬드의 가격을 뜻한다.

### 분석 질문

- **Q1.** 캐럿·컷·색상·투명도를 비롯한 다이아몬드 특성 중 가격과 가장 뚜렷한 관련이 있는 요인은 무엇인가?
- **Q2.** 캐럿이 비슷한 다이아몬드라도 컷·색상·투명도에 따라 가격 차이가 나타나는가?
- **Q3.** 기본 선형회귀와 가격의 로그 변환·캐럿의 제곱항·캐럿과 품질 특성의 상호작용을 반영한 모형을 비교했을 때, 어떤 모형이 다이아몬드 가격을 더 잘 설명하고 예측하는가?

**종속변수:** `price` — 다이아몬드 가격(연속형, 달러 단위) → 회귀 문제

**설명변수:** `carat`, `cut`, `color`, `clarity`, `depth`, `table`, `x`, `y`, `z`

### 이번 노트북에서 연습할 기법 — 선형회귀

데이터의 구조와 품질을 점검하고, 탐색적 분석을 통해 각 변수의 분포와 변수 간 관계를 확인한다. 이후 여러 설명변수를 함께 사용하는 다중선형회귀를 적용하여 다른 조건이 같을 때 각 다이아몬드 특성이 가격과 어떤 관계를 보이는지 살펴본다.

구체적인 자료형 변환, 변수 변환 및 모형 구성 방법은 데이터의 실제 분포와 탐색적 분석 결과를 확인한 뒤 결정한다.

#### 1-1. 분석 환경 설정

> 분석에 필요한 개별 함수 **import**

In [1]:
import sys

portfolio_path = r"C:\Data-Analysis-Portfolio"

if portfolio_path not in sys.path:
    sys.path.insert(0, portfolio_path)

from jihyelab import upload_data

In [2]:
from helpers import *
from IPython.display import display, Markdown

import pandas as pd
import numpy as np
import statsmodels.api as sm

---
## #2. 데이터 소개

#### 2-1. 데이터 로딩

In [3]:
# 캐글데이터 주소 모음집
data = upload_data(
    "Diamonds",
    local=r"C:\Data-Analysis-Portfolio\dataset_repository",
)

data.head()

📚 다이아몬드의 중량·커팅 품질·색상·투명도·크기 등의 특성을 바탕으로 가격을 분석하고 예측하는 데이터셋 (출처: https://www.kaggle.com/datasets/shivam2503/diamonds)

    field    description
--  -------  ------------------------------------------------------------
 0  carat    다이아몬드 중량(캐럿)
 1  cut      다이아몬드 커팅 품질 (Fair, Good, Very Good, Premium, Ideal)
 2  color    다이아몬드 색상 등급 (J부터 D까지이며 D가 가장 높은 등급)
 3  clarity  다이아몬드 투명도 등급
 4  depth    다이아몬드 전체 깊이 비율
 5  table    다이아몬드 상단면의 너비 비율
 6  price    다이아몬드 가격(미국 달러)
 7  x        다이아몬드 길이(mm)
 8  y        다이아몬드 너비(mm)
 9  z        다이아몬드 깊이(mm)



,Unnamed: 0,carat,cut,color,clarity,depth,table,price,x,y,z
0,1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75
